# EUR/USD Trading Agent - Backtest with Oanda Data

This notebook implements a Reinforcement Learning trading agent for EUR/USD using real data from Oanda.

## Setup Steps:
1. Install dependencies
2. Configure Oanda API credentials
3. Fetch historical data
4. Train the agent
5. Evaluate performance

## 1. Install Dependencies

In [3]:
!pip install 'stable-baselines3[extra]' pandas-ta oandapyV20 gymnasium matplotlib -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 1.1.0 requires websockets>=13.0, but you have websockets 12.0 which is incompatible.
langchain 0.3.19 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 2.5.0 which is incompatible.
mlflow 2.20.1 requires pandas<3, but you have pandas 3.0.0 which is incompatible.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.4.2 which is incompatible.
vectorbt 0.28.4 requires pandas<3.0,>=2.0, but you have pandas 3.0.0 which is incompatible.
torchvision 0.23.0 requires torch==2.8.0, but you have torch 2.5.1 which is incompatible.
scikit-learn 1.3.2 requires numpy<2.0,>=1.17.3, but you have numpy 2.2.6 which is incompatible.
streamlit 1.42.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.0 which is incompatible.


## 2. Import Libraries

In [4]:
import os
import numpy as np
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

try:
    import gymnasium as gym
    from gymnasium import spaces
    _GYMNASIUM = True
except ImportError:
    import gym
    from gym import spaces
    _GYMNASIUM = False

from oandapyV20 import API
from oandapyV20.endpoints.instruments import InstrumentsCandles

print("All libraries imported successfully!")

All libraries imported successfully!


## 3. Oanda API Configuration

**IMPORTANT:** You need to:
1. Create a free account at https://www.oanda.com/
2. Get your API token from the account dashboard
3. Enter your credentials below

In [6]:
# ===== ENTER YOUR OANDA CREDENTIALS HERE =====
OANDA_API_TOKEN = "eca5daf33e795e0fd1912fa8acd59df3-b0878fbd73797c1b1706922430cc419b"  # Replace with your actual token
OANDA_ACCOUNT_TYPE = "practice"  # Use 'practice' for demo account, 'live' for real account
# =============================================

if OANDA_API_TOKEN == "YOUR_OANDA_API_TOKEN_HERE":
    print("⚠️ WARNING: Please set your OANDA_API_TOKEN above!")
else:
    print("✓ API credentials configured")

✓ API credentials configured


## 4. Data Fetching Function

In [7]:
def fetch_oanda_data(
    api_token,
    instrument="EUR_USD",
    granularity="H1",  # H1 = Hourly, M15 = 15 minutes, D = Daily, etc.
    count=5000,  # Maximum number of candles to fetch (Oanda limit is 5000)
    from_date=None,
    to_date=None,
    account_type="practice"
):
    """
    Fetch historical OHLCV data from Oanda.
    
    Granularity options:
    - M1, M5, M15, M30 (minutes)
    - H1, H4, H8, H12 (hours)
    - D, W, M (day, week, month)
    """
    api = API(access_token=api_token, environment=account_type)
    
    params = {
        "granularity": granularity,
        "count": count,
        "price": "A"  # Ask prices (use 'M' for mid, 'B' for bid)
    }
    
    if from_date:
        params["from"] = from_date
    if to_date:
        params["to"] = to_date
    
    request = InstrumentsCandles(instrument=instrument, params=params)
    response = api.request(request)
    
    # Parse response
    data = []
    for candle in response['candles']:
        if candle['complete']:
            data.append({
                'Time (EET)': pd.to_datetime(candle['time']),
                'Open': float(candle['ask']['o']),
                'High': float(candle['ask']['h']),
                'Low': float(candle['ask']['l']),
                'Close': float(candle['ask']['c']),
                'Volume': int(candle['volume'])
            })
    
    df = pd.DataFrame(data)
    print(f"✓ Fetched {len(df)} candles from Oanda")
    print(f"  Date range: {df['Time (EET)'].min()} to {df['Time (EET)'].max()}")
    
    return df


def fetch_multiple_batches(
    api_token,
    instrument="EUR_USD",
    granularity="H1",
    total_candles=20000,
    account_type="practice"
):
    """
    Fetch more than 5000 candles by making multiple API calls.
    """
    all_data = []
    batch_size = 5000
    num_batches = (total_candles + batch_size - 1) // batch_size
    
    print(f"Fetching {total_candles} candles in {num_batches} batches...")
    
    to_date = datetime.utcnow()
    
    for i in range(num_batches):
        print(f"\nBatch {i+1}/{num_batches}")
        
        df_batch = fetch_oanda_data(
            api_token=api_token,
            instrument=instrument,
            granularity=granularity,
            count=min(batch_size, total_candles - len(all_data)),
            to_date=to_date.replace(tzinfo=None).isoformat() + 'Z',
            account_type=account_type
        )
        
        if len(df_batch) == 0:
            print("No more data available")
            break
        
        all_data.insert(0, df_batch)  # Insert at beginning (reverse chronological)
        
        # Update to_date to fetch earlier data in next batch
        to_date = df_batch['Time (EET)'].min() - timedelta(seconds=1)
        
        if len(all_data) * batch_size >= total_candles:
            break
    
    # Combine all batches
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=['Time (EET)']).reset_index(drop=True)
    combined_df = combined_df.sort_values('Time (EET)').reset_index(drop=True)
    
    print(f"\n✓ Total candles fetched: {len(combined_df)}")
    return combined_df

## 5. Technical Indicators (indicators.py logic)

In [8]:
def load_and_preprocess_data(df_raw):
    """
    Preprocesses EURUSD data by adding RELATIVE technical features.
    
    The returned DataFrame contains OHLCV for env internals,
    but `feature_cols` lists only the RELATIVE columns to feed the agent.
    """
    df = df_raw.copy()
    
    # Strip any trailing spaces in headers
    df.columns = df.columns.str.strip()
    
    # Set datetime index
    df = df.set_index("Time (EET)")
    df.sort_index(inplace=True)
    
    # Ensure numeric
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # ---- Technical Indicators ----
    # RSI and ATR (scale-invariant)
    df["rsi_14"] = ta.rsi(df["Close"], length=14)
    df["atr_14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)
    
    # Moving averages
    df["ma_20"] = ta.sma(df["Close"], length=20)
    df["ma_50"] = ta.sma(df["Close"], length=50)
    
    # Slopes of the MAs
    df["ma_20_slope"] = df["ma_20"].diff()
    df["ma_50_slope"] = df["ma_50"].diff()
    
    # Distance of price from each MA (relative level)
    df["close_ma20_diff"] = df["Close"] - df["ma_20"]
    df["close_ma50_diff"] = df["Close"] - df["ma_50"]
    
    # MA divergence: MA20 vs MA50
    df["ma_spread"] = df["ma_20"] - df["ma_50"]
    df["ma_spread_slope"] = df["ma_spread"].diff()
    
    # Drop initial NaNs from indicators
    df.dropna(inplace=True)
    
    # Columns the AGENT should see (no raw price levels / raw MAs)
    feature_cols = [
        "rsi_14",
        "atr_14",
        "ma_20_slope",
        "ma_50_slope",
        "close_ma20_diff",
        "close_ma50_diff",
        "ma_spread",
        "ma_spread_slope",
    ]
    
    print(f"✓ Preprocessed data: {len(df)} bars with {len(feature_cols)} features")
    return df, feature_cols

## 6. Trading Environment (trading_env.py logic)

In [9]:
class ForexTradingEnv(gym.Env):
    """
    RL Forex Trading Environment (Position-Persistent)
    
    Key properties:
      - Observation: rolling window of features + 3 state features
      - Actions: HOLD, CLOSE, OPEN (direction + SL + TP)
      - Position persistence until CLOSE or SL/TP hit
      - Friction: spread + commission + optional slippage
    """
    
    metadata = {"render_modes": ["human"]}
    
    def __init__(
        self,
        df,
        window_size: int = 30,
        sl_options=None,
        tp_options=None,
        feature_columns=None,
        pip_value: float = 0.0001,
        spread_pips: float = 1.0,
        commission_pips: float = 0.0,
        max_slippage_pips: float = 0.0,
        lot_size: float = 100000.0,
        reward_scale: float = 1.0,
        unrealized_delta_weight: float = 0.02,
        random_start: bool = True,
        min_episode_steps: int = 300,
        episode_max_steps: int | None = None,
        feature_mean: np.ndarray | None = None,
        feature_std: np.ndarray | None = None,
        allow_flip: bool = False,
        hold_reward_weight: float = 0.005,
        open_penalty_pips: float = 0.5,
        time_penalty_pips: float = 0.02,
    ):
        super().__init__()
        
        self.df = df.reset_index(drop=True)
        self.n_steps = len(self.df)
        
        if feature_columns is None:
            self.feature_columns = list(self.df.columns)
        else:
            self.feature_columns = list(feature_columns)
        
        if sl_options is None or tp_options is None:
            raise ValueError("sl_options and tp_options must be provided")
        self.sl_options = list(sl_options)
        self.tp_options = list(tp_options)
        
        if self.n_steps <= window_size + 2:
            raise ValueError("Dataframe is too short for the given window_size")
        
        self.window_size = int(window_size)
        self.pip_value = float(pip_value)
        
        # Friction
        self.spread_pips = float(spread_pips)
        self.commission_pips = float(commission_pips)
        self.max_slippage_pips = float(max_slippage_pips)
        
        # Equity
        self.lot_size = float(lot_size)
        self.usd_per_pip = self.pip_value * self.lot_size
        
        # Reward
        self.reward_scale = float(reward_scale)
        self.unrealized_delta_weight = float(unrealized_delta_weight)
        self.hold_reward_weight = float(hold_reward_weight)
        self.open_penalty_pips = float(open_penalty_pips)
        self.time_penalty_pips = float(time_penalty_pips)
        
        # Episode
        self.random_start = bool(random_start)
        self.min_episode_steps = int(min_episode_steps)
        self.episode_max_steps = episode_max_steps if episode_max_steps is None else int(episode_max_steps)
        
        # Normalization
        self.feature_mean = feature_mean
        self.feature_std = feature_std
        
        self.allow_flip = bool(allow_flip)
        
        # Actions
        self.action_map = [("HOLD", None, None, None), ("CLOSE", None, None, None)]
        for direction in [0, 1]:  # 0=short, 1=long
            for sl in self.sl_options:
                for tp in self.tp_options:
                    self.action_map.append(("OPEN", direction, float(sl), float(tp)))
        
        self.action_space = spaces.Discrete(len(self.action_map))
        
        # Observation
        self.base_num_features = len(self.feature_columns)
        self.state_num_features = 3
        self.num_features = self.base_num_features + self.state_num_features
        
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.window_size, self.num_features),
            dtype=np.float32
        )
        
        # Internal state
        self._reset_state()
    
    def _reset_state(self):
        self.current_step = 0
        self.steps_in_episode = 0
        self.terminated = False
        self.truncated = False
        
        self.position = 0
        self.entry_price = None
        self.sl_price = None
        self.tp_price = None
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0
        
        self.initial_equity_usd = 10000.0
        self.equity_usd = self.initial_equity_usd
        
        self.equity_curve = []
        self.last_trade_info = None
    
    def _get_state_features(self):
        pos = float(self.position)
        t_norm = float(self.time_in_trade) / 1000.0
        unreal_pips = float(self._compute_unrealized_pips()) if self.position != 0 else 0.0
        unreal_scaled = unreal_pips / 100.0
        return np.array([pos, t_norm, unreal_scaled], dtype=np.float32)
    
    def _compute_unrealized_pips(self):
        if self.position == 0 or self.entry_price is None:
            return 0.0
        close_price = float(self.df.loc[self.current_step, "Close"])
        if self.position == 1:
            pnl_price = close_price - self.entry_price
        else:
            pnl_price = self.entry_price - close_price
        return pnl_price / self.pip_value
    
    def _apply_optional_normalization(self, obs: np.ndarray) -> np.ndarray:
        if self.feature_mean is None or self.feature_std is None:
            return obs
        mean = self.feature_mean.reshape(1, 1, -1)
        std = self.feature_std.reshape(1, 1, -1)
        std = np.where(std == 0, 1.0, std)
        return (obs - mean) / std
    
    def _get_observation(self):
        start = self.current_step - self.window_size
        if start < 0:
            start = 0
        
        obs_df = self.df.iloc[start:self.current_step].copy()
        obs_df = obs_df[self.feature_columns]
        
        if len(obs_df) == 0:
            base = np.tile(self.df.iloc[0].values.astype(np.float32), (self.window_size, 1))
        else:
            base = obs_df.values.astype(np.float32)
            if base.shape[0] < self.window_size:
                pad_rows = self.window_size - base.shape[0]
                pad = np.tile(base[0], (pad_rows, 1))
                base = np.vstack([pad, base])
        
        state_feat = self._get_state_features()
        state_block = np.tile(state_feat, (self.window_size, 1))
        obs = np.hstack([base, state_block]).astype(np.float32)
        
        obs = self._apply_optional_normalization(obs)
        return obs
    
    def _sample_slippage_pips(self) -> float:
        if self.max_slippage_pips <= 0:
            return 0.0
        return float(np.random.uniform(0.0, self.max_slippage_pips))
    
    def _cost_pips_round_trip(self) -> float:
        return self.spread_pips + self.commission_pips
    
    def _open_position(self, direction: int, sl_pips: float, tp_pips: float):
        close_price = float(self.df.loc[self.current_step, "Close"])
        slip_pips = self._sample_slippage_pips()
        slip_price = slip_pips * self.pip_value
        
        if direction == 1:  # long
            entry = close_price + slip_price
            sl_price = entry - sl_pips * self.pip_value
            tp_price = entry + tp_pips * self.pip_value
            self.position = 1
        else:  # short
            entry = close_price - slip_price
            sl_price = entry + sl_pips * self.pip_value
            tp_price = entry - tp_pips * self.pip_value
            self.position = -1
        
        self.entry_price = entry
        self.sl_price = sl_price
        self.tp_price = tp_price
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0
        
        self.last_trade_info = {
            "event": "OPEN",
            "step": self.current_step,
            "position": self.position,
            "entry_price": self.entry_price,
            "sl_price": self.sl_price,
            "tp_price": self.tp_price
        }
    
    def _close_position(self, reason: str, exit_price: float):
        if self.position == 1:
            pnl_price = exit_price - self.entry_price
        else:
            pnl_price = self.entry_price - exit_price
        realized_pips = pnl_price / self.pip_value
        
        cost_pips = self._cost_pips_round_trip()
        net_pips = realized_pips - cost_pips
        
        self.equity_usd += net_pips * self.usd_per_pip
        
        trade_info = {
            "event": "CLOSE",
            "reason": reason,
            "step": self.current_step,
            "position": self.position,
            "entry_price": self.entry_price,
            "exit_price": exit_price,
            "realized_pips": float(realized_pips),
            "cost_pips": float(cost_pips),
            "net_pips": float(net_pips),
            "equity_usd": float(self.equity_usd),
            "time_in_trade": int(self.time_in_trade),
        }
        
        self.position = 0
        self.entry_price = None
        self.sl_price = None
        self.tp_price = None
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0
        
        self.last_trade_info = trade_info
        return net_pips
    
    def _check_sl_tp_intrabar_and_maybe_close(self) -> float:
        if self.position == 0:
            return None
        
        if self.current_step >= self.n_steps - 2:
            exit_price = float(self.df.loc[self.current_step, "Close"])
            net_pips = self._close_position("END_OF_DATA", exit_price)
            return net_pips
        
        next_high = float(self.df.loc[self.current_step + 1, "High"])
        next_low = float(self.df.loc[self.current_step + 1, "Low"])
        
        if self.position == 1:
            sl_hit = next_low <= self.sl_price
            tp_hit = next_high >= self.tp_price
            if sl_hit and tp_hit:
                return self._close_position("SL_AND_TP_SAME_BAR_SL_FIRST", self.sl_price)
            elif sl_hit:
                return self._close_position("SL_HIT", self.sl_price)
            elif tp_hit:
                return self._close_position("TP_HIT", self.tp_price)
        else:
            sl_hit = next_high >= self.sl_price
            tp_hit = next_low <= self.tp_price
            if sl_hit and tp_hit:
                return self._close_position("SL_AND_TP_SAME_BAR_SL_FIRST", self.sl_price)
            elif sl_hit:
                return self._close_position("SL_HIT", self.sl_price)
            elif tp_hit:
                return self._close_position("TP_HIT", self.tp_price)
        
        return None
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self._reset_state()
        
        if self.random_start:
            max_start = self.n_steps - max(self.min_episode_steps, self.window_size) - 2
            if max_start <= self.window_size:
                self.current_step = self.window_size
            else:
                self.current_step = int(np.random.randint(self.window_size, max_start))
        else:
            self.current_step = self.window_size
        
        self.steps_in_episode = 0
        self.terminated = False
        self.truncated = False
        
        obs = self._get_observation()
        
        if _GYMNASIUM:
            return obs, {}
        return obs
    
    def step(self, action: int):
        if self.terminated or self.truncated:
            obs = self._get_observation()
            if _GYMNASIUM:
                return obs, 0.0, True, False, {}
            return obs, 0.0, True, {}
        
        self.steps_in_episode += 1
        reward_pips = 0.0
        info = {}
        
        act_type, direction, sl_pips, tp_pips = self.action_map[int(action)]
        
        # Apply action
        if act_type == "HOLD":
            pass
        elif act_type == "CLOSE":
            if self.position != 0:
                close_price = float(self.df.loc[self.current_step, "Close"])
                slip_pips = self._sample_slippage_pips()
                slip_price = slip_pips * self.pip_value
                exit_price = close_price - slip_price if self.position == 1 else close_price + slip_price
                reward_pips += self._close_position("MANUAL_CLOSE", exit_price)
        elif act_type == "OPEN":
            if self.position == 0:
                self._open_position(direction=direction, sl_pips=sl_pips, tp_pips=tp_pips)
                reward_pips -= self.open_penalty_pips
            else:
                if self.allow_flip:
                    close_price = float(self.df.loc[self.current_step, "Close"])
                    reward_pips += self._close_position("FLIP_CLOSE", close_price)
                    self._open_position(direction=direction, sl_pips=sl_pips, tp_pips=tp_pips)
                    reward_pips -= self.open_penalty_pips
        
        # Check SL/TP
        realized_now = self._check_sl_tp_intrabar_and_maybe_close()
        if realized_now is not None:
            reward_pips += realized_now
        
        # Reward shaping
        if self.position != 0:
            self.time_in_trade += 1
            
            unreal_now = self._compute_unrealized_pips()
            delta_unreal = unreal_now - self.prev_unrealized_pips
            
            if unreal_now > 0:
                reward_pips += self.hold_reward_weight * unreal_now
            
            if self.unrealized_delta_weight != 0.0:
                reward_pips += self.unrealized_delta_weight * delta_unreal
            
            reward_pips -= self.time_penalty_pips
            
            self.prev_unrealized_pips = unreal_now
        
        # Advance time
        self.current_step += 1
        
        # Termination
        if self.current_step >= self.n_steps - 1:
            self.terminated = True
        
        if self.episode_max_steps is not None and self.steps_in_episode >= self.episode_max_steps:
            self.truncated = True
        
        # Log equity
        self.equity_curve.append(float(self.equity_usd))
        
        # Observation
        obs = self._get_observation()
        
        # Final reward
        reward = float(reward_pips) * self.reward_scale
        
        # Info
        info.update({
            "equity_usd": float(self.equity_usd),
            "position": int(self.position),
            "time_in_trade": int(self.time_in_trade),
            "reward_pips": float(reward_pips),
            "last_trade_info": self.last_trade_info
        })
        
        if _GYMNASIUM:
            return obs, reward, self.terminated, self.truncated, info
        else:
            done = bool(self.terminated or self.truncated)
            return obs, reward, done, info
    
    def render(self):
        print(
            f"Step={self.current_step} | Equity=${self.equity_usd:,.2f} | "
            f"Pos={self.position} | Entry={self.entry_price} | SL={self.sl_price} | TP={self.tp_price}"
        )

print("✓ ForexTradingEnv class defined")

✓ ForexTradingEnv class defined


## 7. Fetch Data from Oanda

In [12]:
# Fetch hourly data (adjust granularity and total_candles as needed)
# Options: M15, M30, H1, H4, D

df_raw = fetch_multiple_batches(
    api_token=OANDA_API_TOKEN,
    instrument="EUR_USD",
    granularity="H1",  # Hourly data
    total_candles=20000,  # Fetch ~20k candles
    account_type=OANDA_ACCOUNT_TYPE
)

print(f"\nRaw data shape: {df_raw.shape}")
print(df_raw.head())

Fetching 20000 candles in 4 batches...

Batch 1/4


/var/folders/53/1jv4ycnn6pz77j31lz0263jc0000gn/T/ipykernel_3638/549214540.py:70: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  to_date = datetime.utcnow()


✓ Fetched 4999 candles from Oanda
  Date range: 2025-04-23 22:00:00+00:00 to 2026-02-12 05:00:00+00:00

Batch 2/4


V20Error: {"errorMessage":"Invalid value specified for 'to'"}

## 8. Preprocess Data with Technical Indicators

In [ ]:
df, feature_cols = load_and_preprocess_data(df_raw)

print(f"\nProcessed data shape: {df.shape}")
print(f"Features: {feature_cols}")
print(df.head())

## 9. Split Data: Train/Test (80/20)

In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f"Training bars: {len(train_df)}")
print(f"Testing bars:  {len(test_df)}")

## 10. Training Configuration

In [ ]:
# Environment parameters
SL_OPTS = [5, 10, 15, 25, 30, 60, 90, 120]
TP_OPTS = [5, 10, 15, 25, 30, 60, 90, 120]
WIN = 30

# Training environment (with random starts)
def make_train_env():
    return ForexTradingEnv(
        df=train_df,
        window_size=WIN,
        sl_options=SL_OPTS,
        tp_options=TP_OPTS,
        spread_pips=1.0,
        commission_pips=0.0,
        max_slippage_pips=0.2,
        random_start=True,
        min_episode_steps=1000,
        episode_max_steps=2000,
        feature_columns=feature_cols,
        hold_reward_weight=0.0,
        open_penalty_pips=0.0,
        time_penalty_pips=0.0,
        unrealized_delta_weight=0.0
    )

# Train evaluation environment (deterministic)
def make_train_eval_env():
    return ForexTradingEnv(
        df=train_df,
        window_size=WIN,
        sl_options=SL_OPTS,
        tp_options=TP_OPTS,
        spread_pips=1.0,
        commission_pips=0.0,
        max_slippage_pips=0.2,
        random_start=False,
        episode_max_steps=None,
        feature_columns=feature_cols,
        hold_reward_weight=0.0,
        open_penalty_pips=0.0,
        time_penalty_pips=0.0,
        unrealized_delta_weight=0.0
    )

# Test evaluation environment (deterministic)
def make_test_eval_env():
    return ForexTradingEnv(
        df=test_df,
        window_size=WIN,
        sl_options=SL_OPTS,
        tp_options=TP_OPTS,
        spread_pips=1.0,
        commission_pips=0.0,
        max_slippage_pips=0.2,
        random_start=False,
        episode_max_steps=None,
        feature_columns=feature_cols,
        hold_reward_weight=0.0,
        open_penalty_pips=0.0,
        time_penalty_pips=0.0,
        unrealized_delta_weight=0.0
    )

train_vec_env = DummyVecEnv([make_train_env])
train_eval_env = DummyVecEnv([make_train_eval_env])
test_eval_env = DummyVecEnv([make_test_eval_env])

print("✓ Environments created")

## 11. Evaluation Function

In [ ]:
def evaluate_model(model, eval_env, deterministic=True):
    """Run one episode and return equity curve and final equity."""
    obs = eval_env.reset()
    equity_curve = []
    
    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        step_out = eval_env.step(action)
        
        if len(step_out) == 4:
            obs, rewards, dones, infos = step_out
            done = bool(dones[0])
        else:
            obs, rewards, terminated, truncated, infos = step_out
            done = bool(terminated[0] or truncated[0])
        
        info = infos[0] if isinstance(infos, (list, tuple)) else infos
        eq = info.get("equity_usd", eval_env.get_attr("equity_usd")[0])
        equity_curve.append(eq)
        
        if done:
            break
    
    final_equity = float(equity_curve[-1])
    return equity_curve, final_equity

print("✓ Evaluation function defined")

## 12. Train the Agent

In [ ]:
# Create PPO model
model = PPO(
    policy="MlpPolicy",
    env=train_vec_env,
    verbose=1,
    tensorboard_log="./tensorboard_log/"
)

# Setup checkpoints
ckpt_dir = "./checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=ckpt_dir,
    name_prefix="ppo_eurusd"
)

# Train
total_timesteps = 600_000
print(f"\nStarting training for {total_timesteps:,} timesteps...")
print("This may take 10-30 minutes depending on your hardware.\n")

model.learn(total_timesteps=total_timesteps, callback=checkpoint_callback)

print("\n✓ Training complete!")

## 13. Select Best Model Based on Test Performance

In [ ]:
# Evaluate last model
equity_curve_test_last, final_equity_test_last = evaluate_model(model, test_eval_env)
print(f"[OOS Eval] Last model final equity: ${final_equity_test_last:,.2f}")

# Check all checkpoints
best_equity = -np.inf
best_path = None

ckpts = sorted(
    [f for f in os.listdir(ckpt_dir) if f.endswith(".zip") and f.startswith("ppo_eurusd")],
    key=lambda x: os.path.getmtime(os.path.join(ckpt_dir, x))
)

for ck in ckpts:
    ck_path = os.path.join(ckpt_dir, ck)
    try:
        m = PPO.load(ck_path, env=test_eval_env)
        _, final_eq = evaluate_model(m, test_eval_env)
        print(f"[OOS Eval] {ck} -> final equity: ${final_eq:,.2f}")
        if final_eq > best_equity:
            best_equity = final_eq
            best_path = ck_path
    except Exception as e:
        print(f"[Skip] Could not evaluate checkpoint {ck}: {e}")

# Select best model
if best_path is None or final_equity_test_last >= best_equity:
    print("\nUsing last model as best (by OOS final equity).")
    best_model = model
else:
    print(f"\nUsing best checkpoint: {best_path} (OOS final equity: ${best_equity:,.2f})")
    best_model = PPO.load(best_path, env=train_vec_env)

best_model.save("model_eurusd_best")
print("\n✓ Best model saved: model_eurusd_best")

## 14. Final Evaluation and Visualization

In [ ]:
# Evaluate on both train and test sets
equity_curve_train, final_equity_train = evaluate_model(best_model, train_eval_env)
equity_curve_test, final_equity_test = evaluate_model(best_model, test_eval_env)

print(f"[In-Sample]  Final equity (train): ${final_equity_train:,.2f}")
print(f"[Out-of-Sample] Final equity (test): ${final_equity_test:,.2f}")

# Calculate returns
initial_equity = 10000.0
train_return = ((final_equity_train - initial_equity) / initial_equity) * 100
test_return = ((final_equity_test - initial_equity) / initial_equity) * 100

print(f"\n[In-Sample] Return: {train_return:.2f}%")
print(f"[Out-of-Sample] Return: {test_return:.2f}%")

# Plot equity curves
plt.figure(figsize=(14, 7))
plt.plot(equity_curve_train, label=f"Train (IS) - Return: {train_return:.2f}%", alpha=0.8)
plt.plot(equity_curve_test, label=f"Test (OOS) - Return: {test_return:.2f}%", alpha=0.8)
plt.axhline(y=initial_equity, color='gray', linestyle='--', alpha=0.5, label='Initial Equity')
plt.title("Equity Curves: In-Sample vs Out-of-Sample (Best Model)", fontsize=14, fontweight='bold')
plt.xlabel("Steps")
plt.ylabel("Equity ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Generate Trade History

In [ ]:
def run_one_episode_with_trades(model, vec_env, deterministic=True):
    """Run episode and collect all closed trades."""
    obs = vec_env.reset()
    equity_curve = []
    closed_trades = []
    
    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        step_out = vec_env.step(action)
        
        if len(step_out) == 4:
            obs, rewards, dones, infos = step_out
            done = bool(dones[0])
        else:
            obs, rewards, terminated, truncated, infos = step_out
            done = bool(terminated[0] or truncated[0])
        
        equity_curve.append(vec_env.get_attr("equity_usd")[0])
        
        trade_info = vec_env.get_attr("last_trade_info")[0]
        if isinstance(trade_info, dict) and trade_info.get("event") == "CLOSE":
            closed_trades.append(trade_info)
        
        if done:
            break
    
    return equity_curve, closed_trades

# Get trade history from test set
equity_curve, closed_trades = run_one_episode_with_trades(best_model, test_eval_env, deterministic=True)

if closed_trades:
    trades_df = pd.DataFrame(closed_trades)
    trades_df.to_csv("trade_history_output.csv", index=False)
    print(f"\n✓ Closed trade history saved to trade_history_output.csv")
    print(f"\nTotal trades: {len(trades_df)}")
    print(f"Winning trades: {len(trades_df[trades_df['net_pips'] > 0])}")
    print(f"Losing trades: {len(trades_df[trades_df['net_pips'] < 0])}")
    
    if len(trades_df) > 0:
        win_rate = (len(trades_df[trades_df['net_pips'] > 0]) / len(trades_df)) * 100
        avg_win = trades_df[trades_df['net_pips'] > 0]['net_pips'].mean() if len(trades_df[trades_df['net_pips'] > 0]) > 0 else 0
        avg_loss = trades_df[trades_df['net_pips'] < 0]['net_pips'].mean() if len(trades_df[trades_df['net_pips'] < 0]) > 0 else 0
        
        print(f"\nWin rate: {win_rate:.2f}%")
        print(f"Avg win: {avg_win:.2f} pips")
        print(f"Avg loss: {avg_loss:.2f} pips")
        
        # Display first few trades
        print("\nFirst 10 trades:")
        print(trades_df[['step', 'position', 'realized_pips', 'net_pips', 'reason']].head(10))
else:
    print("\n⚠️ No closed trades recorded.")

## 16. Performance Metrics

In [ ]:
if closed_trades:
    # Calculate additional metrics
    equity_series = pd.Series(equity_curve_test)
    
    # Max drawdown
    running_max = equity_series.expanding().max()
    drawdown = (equity_series - running_max) / running_max * 100
    max_drawdown = drawdown.min()
    
    # Sharpe ratio (simplified - assumes daily returns)
    returns = equity_series.pct_change().dropna()
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    
    print("\n" + "="*50)
    print("PERFORMANCE METRICS (Out-of-Sample)")
    print("="*50)
    print(f"Initial Equity:     ${initial_equity:,.2f}")
    print(f"Final Equity:       ${final_equity_test:,.2f}")
    print(f"Total Return:       {test_return:.2f}%")
    print(f"Max Drawdown:       {max_drawdown:.2f}%")
    print(f"Sharpe Ratio:       {sharpe:.2f}")
    print(f"Total Trades:       {len(trades_df)}")
    print(f"Win Rate:           {win_rate:.2f}%")
    print(f"Avg Win:            {avg_win:.2f} pips")
    print(f"Avg Loss:           {avg_loss:.2f} pips")
    print("="*50)
    
    # Plot drawdown
    plt.figure(figsize=(14, 5))
    plt.plot(drawdown, color='red', alpha=0.7)
    plt.fill_between(range(len(drawdown)), drawdown, 0, alpha=0.3, color='red')
    plt.title("Drawdown Over Time (Test Set)", fontsize=14, fontweight='bold')
    plt.xlabel("Steps")
    plt.ylabel("Drawdown (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 17. Download Files (Optional)

Download the model and trade history to your local machine.

In [ ]:
from google.colab import files

# Download model
if os.path.exists("model_eurusd_best.zip"):
    files.download("model_eurusd_best.zip")
    print("✓ Model downloaded")

# Download trade history
if os.path.exists("trade_history_output.csv"):
    files.download("trade_history_output.csv")
    print("✓ Trade history downloaded")